# Realism — Output 4-1 / 4-2

Edit Realism / Imported data / Macro in Excel, save, then `load_core` +
`load_realism`.

See `docs/04-realism.qmd`.


In [ ]:
from __future__ import annotations

from pathlib import Path

from lic_dsf.load import load_core, load_realism
from lic_dsf.output import (
    fiscal_adjustment_panel,
    fiscal_multiplier_panel,
    forecast_error_panel,
    invest_growth_panel,
    placement_summary,
)
from lic_dsf.realism import (
    place_in_lic_histogram,
    projected_three_year_adjustment,
    rebase_ratio_to_outturn_gdp,
)

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "demo":
    REPO_ROOT = REPO_ROOT.parent
WORKBOOK = REPO_ROOT / "data" / "lic-dsf-template-2025-08-12.xlsx"
WORKBOOK


In [ ]:
# Quick path: compute every Output 4 panel.
from lic_dsf.run import (
    SHEET_4_1,
    SHEET_4_2_FISCAL_ADJ,
    SHEET_4_2_INVEST,
    SHEET_4_2_MULTIPLIER,
    compute_outputs,
)

output_4 = compute_outputs(
    WORKBOOK,
    include=[
        SHEET_4_1,
        SHEET_4_2_FISCAL_ADJ,
        SHEET_4_2_MULTIPLIER,
        SHEET_4_2_INVEST,
    ],
)
list(output_4.sheets)

In [ ]:
macro, external, ext_base, pub_base = load_core(WORKBOOK)
realism = load_realism(WORKBOOK)
first = macro.inputs.first_projection_year
first


## Output 4-2 — Fiscal adjustment


In [ ]:
pd_pct = pub_base.primary_deficit_to_gdp()
projected = projected_three_year_adjustment(pd_pct, first)
print(placement_summary(place_in_lic_histogram(projected)))
out_4_2_adj = fiscal_adjustment_panel(pd_pct, first)
out_4_2_adj.head()


## Output 4-2 — Fiscal multiplier


In [ ]:
pb_pct = 100.0 * macro.primary_balance() / macro.gdp_lcu()
out_4_2_mult = fiscal_multiplier_panel(pb_pct, macro.real_gdp_growth(), first)
out_4_2_mult.loc[:, ("impact", 0.2)].head()


## Output 4-2 — Invest / growth


In [ ]:
out_4_2_invest = invest_growth_panel(
    realism.invest_growth,
    macro.real_gdp_growth().reindex(realism.invest_growth.index),
    realism.capital,
)
out_4_2_invest.head()


## Output 4-1 — Forecast error


In [ ]:
imported = realism.imported
prior_ppg = imported.get("D_PPG_GDP", 2019) or imported.get("D_PPG_GDP", "2019")
current = pub_base.ppg_external_debt_to_gdp()

if prior_ppg is None:
    out_4_1 = None
    print("No D_PPG_GDP vintage in Imported data")
else:
    prior_gdp = imported.get("NGDPD", prior_ppg.vintage_year)
    curr_gdp = imported.get("NGDPD", imported.current_vintage_year)
    if prior_gdp is not None and curr_gdp is not None:
        prior = rebase_ratio_to_outturn_gdp(
            prior_gdp.values, curr_gdp.values, prior_ppg.values
        )
    else:
        prior = prior_ppg.values
        print("NGDPD vintage missing; using unrebased prior D_PPG_GDP")

    years = sorted(set(prior.dropna().index) & set(current.dropna().index))
    out_4_1 = forecast_error_panel(current.reindex(years), prior.reindex(years))

out_4_1
